# 02 — Data Preprocessing — Axe 2 : Stroke Severity (Version 1)

**Projet :** Stroke Prediction AI — IST Dataset  
**Notebook :** 02 — Preprocessing  
**Version :** V1 — Baseline complète (avec RCONSC + RDEF)  
**Dataset :** `IST_corrected.csv` (19 435 patients, 112 variables)  
**Auteur :** ML_Project 2025-2026

---

## Objectif de ce notebook

Ce notebook prépare le jeu de données pour la **modélisation de la sévérité de l'AVC (Axe 2)**.  
Il produit des matrices `X_train`, `X_test`, `y_train`, `y_test` propres, sans fuite d'information, prêtes pour le notebook de modélisation.

### Ce que ce notebook fait, étape par étape :
1. Configuration du projet et chargement centralisé via `config.py`
2. Construction de la cible `severity_class` (variable non présente dans le dataset brut)
3. Sélection des features baseline disponibles à l'admission
4. Définition des deux versions de features (V1 complète / V2 stricte)
5. Split train/test stratifié (80/20)
6. Construction du pipeline de preprocessing (imputation + encodage)
7. Transformation sans fuite d'information (fit sur train uniquement)
8. Sauvegarde de tous les artefacts pour le notebook 03

### Méthodologie V1
La **Version 1** utilise **toutes les features baseline**, y compris `RCONSC` et `RDEF1..RDEF8`  
qui entrent dans la construction de la cible. Cela permet de :
- établir une baseline de performance,
- détecter un éventuel **data leakage structurel**,
- comparer avec la V2 (qui exclut ces variables).

> ⚠️ La V1 est volontairement non-corrigée pour le leakage — c'est un choix méthodologique  
> documenté, qui sert de référence pour la démonstration dans le notebook de modélisation.

---


---
## Section 1 — Configuration du projet

### 1.1 Pourquoi centraliser la configuration ?

Dans un projet ML multi-notebooks, il est essentiel que tous les notebooks partagent :
- les **mêmes chemins** vers les données et artefacts,
- la **même graine aléatoire** (`RANDOM_STATE`) pour la reproductibilité,
- les **mêmes listes de variables** sélectionnées lors de l'EDA.

Le fichier `src/config.py` centralise tout cela.  
Il est importé ici et dans le notebook de modélisation, garantissant la cohérence du pipeline.

### 1.2 Structure du projet sur Google Drive
```
ML_Project/
├── Data/
│   └── IST_corrected.csv
├── src/
│   ├── config.py          ← configuration centralisée
│   └── axe2_utils.py      ← fonctions utilitaires Axe 2
├── artifacts/
│   └── axe2/              ← artefacts produits par ce notebook
│       ├── df_axe2_clean.csv
│       ├── X_train_preprocessed.csv
│       ├── X_test_preprocessed.csv
│       ├── y_train.csv
│       ├── y_test.csv
│       ├── preprocessor_v1.pkl
│       └── preprocessing_metadata_v1.json
└── Notebooks/
    └── Version1/
        ├── 02_Preprocessing_V1.ipynb   ← ce notebook
        └── 03_Modeling_V1.ipynb
```


In [3]:
# ============================================================
# 1.1 — Imports standards
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

print("Imports standards chargés")


Imports standards chargés


In [4]:
# ============================================================
# 1.2 — Montage Google Drive + ajout du chemin src au PATH
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# Racine du projet sur Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project")
SRC_PATH     = PROJECT_ROOT / "src"

# Ajout de src au sys.path pour pouvoir importer config et axe2_utils
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SRC_PATH     :", SRC_PATH)
print("Dossier src existe ?", SRC_PATH.exists())


Mounted at /content/drive
PROJECT_ROOT : /content/drive/MyDrive/ML_Project
SRC_PATH     : /content/drive/MyDrive/ML_Project/src
Dossier src existe ? True


In [5]:
# ============================================================
# 1.3 — Import de config.py (configuration centralisée)
# ============================================================

# config.py contient : DATA_PATH, RANDOM_STATE, ARTIFACTS_DIR,
# load_features(), check_data_quality(), ensure_directories()
import importlib
import config
importlib.reload(config)  # forcer le rechargement si modifié

from config import (
    DATA_PATH,        # chemin vers IST_corrected.csv
    RANDOM_STATE,     # graine aléatoire commune (42)
    load_features,    # charge features.json produit par l'EDA
    check_data_quality,  # vérification rapide du dataset
    ensure_directories,  # crée les dossiers manquants
)

print(" config.py importé")
print("   DATA_PATH    :", DATA_PATH)
print("   RANDOM_STATE :", RANDOM_STATE)


 config.py importé
   DATA_PATH    : /content/drive/MyDrive/ML_Project/Data/IST_corrected.csv
   RANDOM_STATE : 42


In [6]:
# ============================================================
# 1.4 — Import de axe2_utils.py (fonctions métier Axe 2)
# ============================================================

# axe2_utils.py contient les fonctions spécifiques à l'Axe 2 :
# - build_severity_target()   : construit severity_class
# - get_feature_versions()    : retourne features V1 et V2
# - encode_rconsc_if_present(): encode RCONSC en ordinal
# - build_preprocessor()      : construit le ColumnTransformer

from axe2_utils import (
    build_severity_target,
    get_feature_versions,
    encode_rconsc_if_present,
    build_preprocessor,
)

# Création des dossiers artefacts si inexistants
ensure_directories()

# Dossier dédié aux artefacts Axe 2
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "axe2"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Variable cible de l'Axe 2
TARGET = "severity_class"

print("axe2_utils.py importé")
print("ARTIFACTS_DIR :", ARTIFACTS_DIR)


axe2_utils.py importé
ARTIFACTS_DIR : /content/drive/MyDrive/ML_Project/artifacts/axe2


---
## Section 2 — Chargement du dataset et des variables EDA

### 2.1 Pourquoi recharger les variables depuis l'EDA ?

Le notebook 01 (EDA) a produit un fichier `features.json` qui liste :
- les variables baseline disponibles à l'admission,
- leur classification (numérique / catégorielle),
- la variable cible.

Recharger ce fichier ici garantit que le preprocessing utilise exactement  
les mêmes variables que celles identifiées lors de l'analyse exploratoire.  
Cela crée une **chaîne de cohérence** entre les notebooks.

### 2.2 Encodage latin1
Le fichier IST contient des caractères spéciaux d'origine clinique.  
L'encodage `latin1` est utilisé pour éviter les erreurs de lecture.


In [7]:
# ============================================================
# 2.1 — Chargement du dataset brut
# ============================================================

df = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)

print("=" * 50)
print("DATASET BRUT")
print("=" * 50)
print(f"  Lignes    : {df.shape[0]:,}")
print(f"  Colonnes  : {df.shape[1]}")
print(f"  Mémoire   : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

# Vérification rapide de la qualité
check_data_quality(df)


DATASET BRUT
  Lignes    : 19,435
  Colonnes  : 112
  Mémoire   : 66.0 MB

📊 DATA QUALITY REPORT

🔎 Valeurs manquantes totales : 379257

Colonnes avec valeurs manquantes :
FLASTD      19377
DRSHD       19338
DPED        19310
DMAJNCHX    19290
DMAJNCHD    19286
DDEADX      19267
DRSUNKD     19175
DRSISCD     19022
DNOSTRKX    19008
DSIDED      18802
DSIDEX      18797
FDEADX      18651
DHH14       18451
NCCODE      17916
DDEADD      17401
DDEADC      17101
FDEADD      15089
FDEADC      15071
DPLACE       9709
DALIVED      9113
FOAC         4574
FAP          4568
FPLACE       4502
FDENNIS      4499
FRECOVER     4420
DRSUNK       1007
DMH14        1006
DCAREND      1005
RATRIAL       984
RASP3         984
FU1_COMP      470
RHEP24        344
DTHROMB       315
DIVH          305
DSCH          305
DASPLT        147
FU2_DONE       99
FDEAD          99
DCAA           29
DSTER          28
DHAEMD         28
DALIVE         28
DDIAGHA        26
DNOSTRK        26
DGORM          23
DDIAGUN        23


np.False_

In [8]:
# ============================================================
# 2.2 — Rechargement des variables identifiées en EDA
# ============================================================

# load_features() lit le fichier features.json produit en fin d'EDA
EDA_TARGET, EDA_CATEGORICAL_VARS, EDA_NUMERICAL_VARS, EDA_ALL_FEATURES = load_features()

print("Variables rechargées depuis features.json :")
print(f"  Cible              : {EDA_TARGET}")
print(f"  Nb variables cat   : {len(EDA_CATEGORICAL_VARS)}")
print(f"  Nb variables num   : {len(EDA_NUMERICAL_VARS)}")
print(f"  Total features     : {len(EDA_ALL_FEATURES)}")
print()
print("Features EDA complètes :")
print(EDA_ALL_FEATURES)



📦 features.json chargé
🔹 target      : stroke
🔹 categorical : 17
🔹 numerical   : 3
Variables rechargées depuis features.json :
  Cible              : stroke
  Nb variables cat   : 17
  Nb variables num   : 3
  Total features     : 20

Features EDA complètes :
['RCONSC', 'SEX', 'RSLEEP', 'RATRIAL', 'RCT', 'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5', 'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'RDELAY', 'AGE', 'RSBP']


---
## Section 3 — Construction de la variable cible `severity_class`

### 3.1 Pourquoi construire la cible ?

La variable `severity_class` n'existe **pas** dans le dataset IST brut.  
Elle est construite à partir d'un **score composite** défini dans le cahier des charges.

### 3.2 Règle de construction

**Étape 1 — Score de déficits neurologiques (RDEF_score)**
```
RDEF1..RDEF8 : Y → 1  |  N → 0  |  C → NaN (non évaluable)
RDEF_score = somme des RDEF_i binaires (sur les valeurs non-NaN)
```

**Étape 2 — Facteur de conscience (RCONSC_factor)**
```
RCONSC : F (Fully conscious) → 0
         D (Drowsy)          → 1
         U (Unconscious)     → 2
```

**Étape 3 — Score final et classification**
```
severity_score = RDEF_score + 2 × RCONSC_factor

Classes :
  0–3  → léger   (AVC de faible impact neurologique)
  4–6  → modéré  (AVC avec déficits multiples ou somnolence)
  7+   → sévère  (AVC grave, patient inconscient ou déficits majeurs)
```

### 3.3 Note sur les "C" dans les RDEF

`C = "can't assess"` signifie que l'évaluateur **ne pouvait pas** tester ce déficit  
(ex : patient inconscient, non coopératif). En V1, ces valeurs sont traitées comme NaN  
avant l'imputation. La V2 améliorera ce traitement.


In [9]:
# ============================================================
# 3.1 — Construction de la cible severity_class
# ============================================================

# build_severity_target() ajoute au DataFrame :
# - RDEF_score       : somme des déficits confirmés (Y=1, N=0, C→NaN)
# - RCONSC_factor    : encodage ordinal de l'état de conscience
# - severity_score   : score composite
# - severity_class   : classe cible (leger / modere / severe)

df = build_severity_target(df)

# Aperçu des variables intermédiaires
print("Aperçu des variables de sévérité construites :")
display(df[["RDEF_score", "RCONSC_factor", "severity_score", "severity_class"]].head(10))

print()
print("Distribution de la cible severity_class :")
dist = df["severity_class"].value_counts(dropna=False)
dist_pct = df["severity_class"].value_counts(normalize=True, dropna=False) * 100
summary = pd.DataFrame({"count": dist, "pct (%)": dist_pct.round(1)})
display(summary)


Aperçu des variables de sévérité construites :


,RDEF_score,RCONSC_factor,severity_score,severity_class
0,3.0,1,5.0,modere
1,3.0,0,3.0,leger
2,3.0,0,3.0,leger
3,1.0,0,1.0,leger
4,3.0,0,3.0,leger
5,3.0,0,3.0,leger
6,4.0,0,4.0,modere
7,1.0,0,1.0,leger
8,1.0,0,1.0,leger
9,2.0,0,2.0,leger



Distribution de la cible severity_class :


,count,pct (%)
severity_class,,
leger,9692,49.9
modere,8096,41.7
severe,1630,8.4
NaN,17,0.1


### 3.4 Interprétation de la distribution

| Classe | Effectif | % | Interprétation clinique |
|--------|----------|---|------------------------|
| **léger** | ~9 700 | ~50% | AVC peu sévère, patient conscient et peu de déficits |
| **modéré** | ~8 100 | ~42% | AVC intermédiaire, somnolence ou déficits multiples |
| **sévère** | ~1 600 | **~8%** | AVC grave, patient inconscient ou nombreux déficits |

>  **Déséquilibre de classes** : la classe `sévère` représente seulement 8% des cas.  
> Le modèle sera tenté de la négliger au profit des classes majoritaires.  
> C'est un challenge central de cet axe — qui sera adressé dans les versions suivantes.


In [10]:
# ============================================================
# 3.2 — Suppression des lignes sans cible calculable
# ============================================================

# Certaines lignes ont trop de NaN dans RDEF ou RCONSC pour calculer severity_score
# → impossible de leur attribuer une classe → on les exclut du dataset de travail

n_before = len(df)
df_model = df.dropna(subset=["severity_class"]).copy()
n_after  = len(df_model)

print(f"Lignes avant suppression NaN cible : {n_before:,}")
print(f"Lignes après suppression NaN cible : {n_after:,}")
print(f"Lignes retirées                    : {n_before - n_after:,} ({(n_before-n_after)/n_before*100:.1f}%)")
print()
print("Distribution finale :")
print(df_model["severity_class"].value_counts())


Lignes avant suppression NaN cible : 19,435
Lignes après suppression NaN cible : 19,418
Lignes retirées                    : 17 (0.1%)

Distribution finale :
severity_class
leger     9692
modere    8096
severe    1630
Name: count, dtype: int64


---
## Section 4 — Sélection des features et gestion du Data Leakage

### 4.1 Principe : n'utiliser que les variables disponibles à l'admission

Pour prédire la sévérité **au moment de l'admission**, le modèle ne peut utiliser  
que les informations collectées à ce moment-là. Utiliser des variables post-admission  
créerait un **data leakage temporel** : le modèle "verrait le futur".

### 4.2 Leakage structurel en V1 — explication pédagogique

En **Version 1**, on conserve intentionnellement `RCONSC` et `RDEF1..RDEF8`  
dans les features, bien qu'elles entrent dans la formule de construction de la cible.

Ce n'est **pas** un leakage temporel (ces variables sont bien disponibles à l'admission),  
mais un **leakage structurel** : le modèle peut reconstruire presque parfaitement  
la cible en "apprenant" la règle de calcul du score.

**Cela se manifeste par :** performances quasi parfaites (F1 > 0.99) en V1.  
**L'objectif pédagogique :** montrer ce phénomène et le corriger dans la comparaison V1 vs V2.

### 4.3 Variables exclues dans V2 (pas dans ce notebook)
```
RCONSC, RDEF1, RDEF2, RDEF3, RDEF4, RDEF5, RDEF6, RDEF7, RDEF8
```


In [11]:
# ============================================================
# 4.1 — Variables baseline candidates (rechargées depuis EDA)
# ============================================================

# On filtre pour ne garder que les colonnes présentes dans df_model
baseline_features = [col for col in EDA_ALL_FEATURES if col in df_model.columns]

print(f"Nombre de features baseline candidates : {len(baseline_features)}")
print()
print("Liste complète :")
for i, f in enumerate(baseline_features, 1):
    print(f"  {i:2d}. {f}")


Nombre de features baseline candidates : 20

Liste complète :
   1. RCONSC
   2. SEX
   3. RSLEEP
   4. RATRIAL
   5. RCT
   6. RVISINF
   7. RHEP24
   8. RASP3
   9. RDEF1
  10. RDEF2
  11. RDEF3
  12. RDEF4
  13. RDEF5
  14. RDEF6
  15. RDEF7
  16. RDEF8
  17. STYPE
  18. RDELAY
  19. AGE
  20. RSBP


In [12]:
# ============================================================
# 4.2 — Définition des versions V1 et V2
# ============================================================

# get_feature_versions() retourne :
# - features_v1 : toutes les baseline features (y compris RCONSC + RDEF)
# - features_v2 : baseline stricte (RCONSC + RDEF exclus)

features_v1, features_v2 = get_feature_versions(baseline_features)

print("=" * 50)
print("VERSION 1 — Baseline complète (avec leakage structurel)")
print("=" * 50)
print(f"  Nb features : {len(features_v1)}")
print(f"  Variables   : {features_v1}")

print()
print("=" * 50)
print("VERSION 2 — Baseline stricte (sans RCONSC ni RDEF)")
print("=" * 50)
print(f"  Nb features : {len(features_v2)}")
print(f"  Variables   : {features_v2}")

print()
print(f"Différence : {set(features_v1) - set(features_v2)}")
print("→ Ce sont les variables retirées en V2 pour corriger le leakage structurel")


VERSION 1 — Baseline complète (avec leakage structurel)
  Nb features : 20
  Variables   : ['RCONSC', 'SEX', 'RSLEEP', 'RATRIAL', 'RCT', 'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5', 'RDEF6', 'RDEF7', 'RDEF8', 'STYPE', 'RDELAY', 'AGE', 'RSBP']

VERSION 2 — Baseline stricte (sans RCONSC ni RDEF)
  Nb features : 11
  Variables   : ['SEX', 'RSLEEP', 'RATRIAL', 'RCT', 'RVISINF', 'RHEP24', 'RASP3', 'STYPE', 'RDELAY', 'AGE', 'RSBP']

Différence : {'RDEF7', 'RDEF2', 'RDEF4', 'RDEF8', 'RDEF5', 'RCONSC', 'RDEF1', 'RDEF6', 'RDEF3'}
→ Ce sont les variables retirées en V2 pour corriger le leakage structurel


In [13]:
# ============================================================
# 4.3 — Construction de X et y pour la Version 1
# ============================================================

# Ce notebook travaille sur la VERSION 1 (baseline complète)
# Pour la V2, il suffit de remplacer features_v1 par features_v2

selected_features = features_v1   # ← VERSION 1 : baseline complète

X = df_model[selected_features].copy()
y = df_model[TARGET].copy()

print("Feature set sélectionné : VERSION 1 (baseline complète)")
print(f"  Shape de X : {X.shape}")
print(f"  Shape de y : {y.shape}")
print()
print("Distribution de y :")
print(y.value_counts())


Feature set sélectionné : VERSION 1 (baseline complète)
  Shape de X : (19418, 20)
  Shape de y : (19418,)

Distribution de y :
severity_class
leger     9692
modere    8096
severe    1630
Name: count, dtype: int64


---
## Section 5 — Analyse des données avant preprocessing

### 5.1 Pourquoi analyser les données avant de transformer ?

Avant d'appliquer le pipeline, il faut comprendre :
- quelles variables sont numériques vs catégorielles,
- combien de valeurs manquantes sont présentes et pour quelles variables,
- si des valeurs aberrantes visibles nécessitent un traitement spécial.

Cette étape permet de **choisir une stratégie d'imputation adaptée** à chaque type de variable.


In [14]:
# ============================================================
# 5.1 — Séparation variables numériques / catégorielles
# ============================================================

numeric_features     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variables NUMÉRIQUES :")
for col in numeric_features:
    print(f"  - {col}")

print()
print("Variables CATÉGORIELLES :")
for col in categorical_features:
    print(f"  - {col}")


Variables NUMÉRIQUES :
  - RDELAY
  - AGE
  - RSBP

Variables CATÉGORIELLES :
  - RCONSC
  - SEX
  - RSLEEP
  - RATRIAL
  - RCT
  - RVISINF
  - RHEP24
  - RASP3
  - RDEF1
  - RDEF2
  - RDEF3
  - RDEF4
  - RDEF5
  - RDEF6
  - RDEF7
  - RDEF8
  - STYPE


In [15]:
# ============================================================
# 5.2 — Analyse des valeurs manquantes dans X
# ============================================================

# On analyse uniquement les variables sélectionnées (pas tout le dataset)
missing_count = X.isna().sum()
missing_pct   = (X.isna().mean() * 100)

missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct (%)": missing_pct.round(2),
    "type": X.dtypes
}).sort_values("missing_count", ascending=False)

# Affichage uniquement des colonnes avec NaN
missing_with_nan = missing_summary[missing_summary["missing_count"] > 0]

if len(missing_with_nan) == 0:
    print(" Aucune valeur manquante dans X")
else:
    print(f"Variables avec valeurs manquantes ({len(missing_with_nan)} colonnes) :")
    display(missing_with_nan)

print()
print("Stratégie d'imputation :")
print("  → Variables numériques  : médiane (robuste aux valeurs extrêmes)")
print("  → Variables catégorielles : mode / most_frequent (modalité la plus fréquente)")


Variables avec valeurs manquantes (3 colonnes) :


,missing_count,missing_pct (%),type
RATRIAL,983,5.06,object
RASP3,983,5.06,object
RHEP24,343,1.77,object



Stratégie d'imputation :
  → Variables numériques  : médiane (robuste aux valeurs extrêmes)
  → Variables catégorielles : mode / most_frequent (modalité la plus fréquente)


### 5.2 Interprétation des valeurs manquantes

Les variables `RATRIAL` et `RASP3` ont environ **5% de NaN** — il s'agit de la phase pilote  
du trial IST (984 patients) où ces données n'étaient pas collectées.  
Ce n'est pas une erreur de saisie, mais une **donnée structurellement manquante**.

La stratégie `most_frequent` est appropriée : elle impute par la modalité dominante,  
ce qui introduit peu de biais dans un contexte clinique.


---
## Section 6 — Encodage ordinal de RCONSC

### 6.1 Pourquoi un encodage ordinal pour RCONSC ?

`RCONSC` représente l'**état de conscience** à l'admission :
- `F` = Fully conscious (pleinement conscient)
- `D` = Drowsy (somnolent)
- `U` = Unconscious (inconscient)

Il existe un **ordre médical naturel** : F < D < U (de moins grave à plus grave).  
L'encodage ordinal `F→0, D→1, U→2` préserve cet ordre,  
ce qui est cliniquement correct et plus informatif qu'un encodage one-hot.

Un encodage one-hot traiterait ces 3 états comme **indépendants** et sans ordre,  
perdant l'information de progression de gravité.


In [16]:
# ============================================================
# 6.1 — Encodage ordinal manuel de RCONSC
# ============================================================

# encode_rconsc_if_present() vérifie si RCONSC est dans X
# et applique le mapping F→0, D→1, U→2
# Si RCONSC est absent (V2), la fonction ne fait rien

X = encode_rconsc_if_present(X)

# Recalcul des listes de types après transformation
numeric_features     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Après encodage de RCONSC :")
print(f"  Variables numériques     : {numeric_features}")
print(f"  Variables catégorielles  : {categorical_features}")

if "RCONSC" in numeric_features:
    print()
    print("Distribution de RCONSC encodée :")
    print(X["RCONSC"].value_counts().sort_index())
    print("  → 0 = F (conscient), 1 = D (somnolent), 2 = U (inconscient)")


Après encodage de RCONSC :
  Variables numériques     : ['RCONSC', 'RDELAY', 'AGE', 'RSBP']
  Variables catégorielles  : ['SEX', 'RSLEEP', 'RATRIAL', 'RCT', 'RVISINF', 'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3', 'RDEF4', 'RDEF5', 'RDEF6', 'RDEF7', 'RDEF8', 'STYPE']

Distribution de RCONSC encodée :
RCONSC
0    14920
1     4252
2      246
Name: count, dtype: int64
  → 0 = F (conscient), 1 = D (somnolent), 2 = U (inconscient)


---
## Section 7 — Séparation Train / Test

### 7.1 Règle fondamentale : splitter AVANT toute transformation

Le split doit être fait **avant** l'imputation, le scaling et l'encodage.  
Si on transforme d'abord puis on splitte, les statistiques du test set  
(médiane, mode, fréquences) influencent l'entraînement → **fuite d'information**.

### 7.2 Pourquoi `stratify=y` ?

La classe `sévère` ne représente que **8%** des données.  
Sans stratification, un tirage aléatoire pourrait aboutir à un test set  
avec très peu (voire zéro) d'exemples sévères.  
`stratify=y` garantit que **les proportions sont conservées** dans train et test.

### 7.3 Ratio 80/20

- **80% train** : suffisant pour entraîner des modèles robustes sur ~15 500 patients
- **20% test** : ~3 900 patients donnent une estimation fiable de la performance généralisée


In [17]:
# ============================================================
# 7.1 — Split train / test stratifié 80/20
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,  # depuis config.py → reproductibilité garantie
    stratify=y                  # conservation des proportions de classes
)

print("=" * 50)
print("RÉSULTAT DU SPLIT")
print("=" * 50)
print(f"  X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"  y_train : {y_train.shape}  |  y_test : {y_test.shape}")

print()
print("Distribution des classes — TRAIN :")
train_dist = y_train.value_counts()
train_pct  = y_train.value_counts(normalize=True) * 100
for cls in ["leger", "modere", "severe"]:
    if cls in train_dist.index:
        print(f"  {cls:8s} : {train_dist[cls]:5d} ({train_pct[cls]:.1f}%)")

print()
print("Distribution des classes — TEST :")
test_dist = y_test.value_counts()
test_pct  = y_test.value_counts(normalize=True) * 100
for cls in ["leger", "modere", "severe"]:
    if cls in test_dist.index:
        print(f"  {cls:8s} : {test_dist[cls]:5d} ({test_pct[cls]:.1f}%)")

print()
print(" La stratification est correcte : proportions similaires train/test")


RÉSULTAT DU SPLIT
  X_train : (15534, 20)  |  X_test : (3884, 20)
  y_train : (15534,)  |  y_test : (3884,)

Distribution des classes — TRAIN :
  leger    :  7753 (49.9%)
  modere   :  6477 (41.7%)
  severe   :  1304 (8.4%)

Distribution des classes — TEST :
  leger    :  1939 (49.9%)
  modere   :  1619 (41.7%)
  severe   :   326 (8.4%)

 La stratification est correcte : proportions similaires train/test


---
## Section 8 — Construction du pipeline de preprocessing

### 8.1 Architecture du ColumnTransformer

On utilise un `ColumnTransformer` sklearn avec **deux sous-pipelines** :

```
ColumnTransformer
├── Pipeline NUMÉRIQUE
│   └── SimpleImputer(strategy='median')
│       → impute les NaN par la médiane de chaque colonne
│       → robuste aux valeurs extrêmes (AGE élevé, RSBP extrême)
│
└── Pipeline CATÉGORIEL
    ├── SimpleImputer(strategy='most_frequent')
    │   → impute les NaN par la modalité la plus fréquente
    └── OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        → encode les modalités en colonnes binaires
        → handle_unknown='ignore' : valeurs inconnues en test → vecteur 0
```

### 8.2 Pourquoi cette architecture ?

Le `ColumnTransformer` applique des transformations différentes selon le type de variable,  
ce qui est la bonne pratique sklearn. Il est encapsulé dans des `Pipeline` pour éviter  
toute fuite entre l'imputation et l'encodage.

### 8.3 Règle d'or : fit sur train, transform sur train ET test

- `preprocessor.fit_transform(X_train)` : apprend les statistiques sur le train, transforme
- `preprocessor.transform(X_test)` : applique les mêmes statistiques au test

Les statistiques du test set (médiane, mode) ne sont **jamais** utilisées.


In [18]:
# ============================================================
# 8.1 — Construction du ColumnTransformer
# ============================================================

# build_preprocessor() construit le ColumnTransformer adapté à X_train
# Il détecte automatiquement les colonnes numériques et catégorielles
preprocessor = build_preprocessor(X_train)

print("Pipeline de preprocessing construit :")
print(preprocessor)


Pipeline de preprocessing construit :
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['RCONSC', 'RDELAY', 'AGE', 'RSBP']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['SEX', 'RSLEEP', 'RATRIAL', 'RCT', 'RVISINF',
                                  'RHEP24', 'RASP3', 'RDEF1', 'RDEF2', 'RDEF3',
                                  'RDEF4', 'RDEF5', 'RDEF6', 'RDEF7', 'RDEF8',
                                  'STYPE'])])


In [19]:
# ============================================================
# 8.2 — Fit sur TRAIN uniquement, transform sur TRAIN et TEST
# ============================================================

# fit_transform sur train : apprend et transforme en même temps
X_train_processed = preprocessor.fit_transform(X_train)

# transform sur test : applique les statistiques apprises sur train
X_test_processed  = preprocessor.transform(X_test)

print("Transformation terminée :")
print(f"  X_train_processed : {X_train_processed.shape}")
print(f"  X_test_processed  : {X_test_processed.shape}")
print()

# Récupération des noms de colonnes après encodage one-hot
feature_names = preprocessor.get_feature_names_out()
print(f"Nombre de features après preprocessing : {len(feature_names)}")
print("Explication : l'augmentation est due au one-hot encoding des variables catégorielles")


Transformation terminée :
  X_train_processed : (15534, 47)
  X_test_processed  : (3884, 47)

Nombre de features après preprocessing : 47
Explication : l'augmentation est due au one-hot encoding des variables catégorielles


In [20]:
# ============================================================
# 8.3 — Conversion en DataFrame lisible
# ============================================================

# Conversion utile pour inspection, debug et lecture de l'output
X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Aperçu de X_train_df après preprocessing :")
display(X_train_df.head(5))

print()
print(f"Shape X_train_df : {X_train_df.shape}")
print(f"Shape X_test_df  : {X_test_df.shape}")


Aperçu de X_train_df après preprocessing :


,num__RCONSC,num__RDELAY,num__AGE,num__RSBP,cat__SEX_F,cat__SEX_M,cat__RSLEEP_N,cat__RSLEEP_Y,cat__RATRIAL_N,cat__RATRIAL_Y,cat__RCT_N,cat__RCT_Y,cat__RVISINF_N,cat__RVISINF_Y,cat__RHEP24_N,cat__RHEP24_Y,cat__RASP3_N,cat__RASP3_Y,cat__RDEF1_C,cat__RDEF1_N,cat__RDEF1_Y,cat__RDEF2_C,cat__RDEF2_N,cat__RDEF2_Y,cat__RDEF3_C,cat__RDEF3_N,cat__RDEF3_Y,cat__RDEF4_C,cat__RDEF4_N,cat__RDEF4_Y,cat__RDEF5_C,cat__RDEF5_N,cat__RDEF5_Y,cat__RDEF6_C,cat__RDEF6_N,cat__RDEF6_Y,cat__RDEF7_C,cat__RDEF7_N,cat__RDEF7_Y,cat__RDEF8_C,cat__RDEF8_N,cat__RDEF8_Y,cat__STYPE_LACS,cat__STYPE_OTH,cat__STYPE_PACS,cat__STYPE_POCS,cat__STYPE_TACS
17221,0.0,17.0,65.0,140.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
14676,1.0,4.0,89.0,130.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
12817,0.0,42.0,84.0,185.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1100,0.0,16.0,73.0,130.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
19018,0.0,7.0,90.0,170.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0



Shape X_train_df : (15534, 47)
Shape X_test_df  : (3884, 47)


In [21]:
# ============================================================
# 8.4 — Vérification finale : absence de valeurs manquantes
# ============================================================

nan_train = X_train_df.isna().sum().sum()
nan_test  = X_test_df.isna().sum().sum()

print("Vérification post-preprocessing :")
print(f"  NaN dans X_train_df : {nan_train}")
print(f"  NaN dans X_test_df  : {nan_test}")

if nan_train == 0 and nan_test == 0:
    print()
    print("Le preprocessing a correctement traité toutes les valeurs manquantes")
else:
    print()
    print("Des NaN subsistent — vérifier le pipeline")


Vérification post-preprocessing :
  NaN dans X_train_df : 0
  NaN dans X_test_df  : 0

Le preprocessing a correctement traité toutes les valeurs manquantes


---
## Section 9 — Encodage de la variable cible

### 9.1 Pourquoi encoder la cible ?

Certains modèles sklearn (notamment `RandomForestClassifier`) acceptent les labels texte.  
Cependant, pour standardiser le pipeline et faciliter l'évaluation, on encode :

```
leger  → 0
modere → 1
severe → 2
```

Cet encodage ordinal **n'implique pas** que le modèle traite les classes comme ordonnées  
(ce n'est pas une régression). Il s'agit simplement d'une convention pour les métriques  
et les visualisations.


In [22]:
# ============================================================
# 9.1 — Encodage de la cible (leger/modere/severe → 0/1/2)
# ============================================================

TARGET_MAP     = {"leger": 0, "modere": 1, "severe": 2}
TARGET_MAP_INV = {0: "leger", 1: "modere", 2: "severe"}

y_train_enc = y_train.map(TARGET_MAP)
y_test_enc  = y_test.map(TARGET_MAP)

print("Distribution y_train encodé :")
print(y_train_enc.value_counts().sort_index().rename(TARGET_MAP_INV))

print()
print("Distribution y_test encodé :")
print(y_test_enc.value_counts().sort_index().rename(TARGET_MAP_INV))

print()
print("Mapping appliqué :", TARGET_MAP)


Distribution y_train encodé :
severity_class
leger     7753
modere    6477
severe    1304
Name: count, dtype: int64

Distribution y_test encodé :
severity_class
leger     1939
modere    1619
severe     326
Name: count, dtype: int64

Mapping appliqué : {'leger': 0, 'modere': 1, 'severe': 2}


---
## Section 10 — Sauvegarde des artefacts

### 10.1 Pourquoi sauvegarder les artefacts ?

Le notebook de modélisation (03) doit utiliser **exactement les mêmes données**  
que celles préparées ici. Sans sauvegarde, il faudrait re-exécuter tout le preprocessing  
dans chaque notebook, ce qui :
- consomme du temps inutilement,
- risque d'introduire des différences si le code est modifié entre deux exécutions.

### 10.2 Artefacts sauvegardés

| Fichier | Contenu |
|---------|---------|
| `df_axe2_clean.csv` | Dataset nettoyé avec cible construite |
| `X_train_preprocessed.csv` | Features d'entraînement transformées |
| `X_test_preprocessed.csv` | Features de test transformées |
| `y_train.csv` | Labels d'entraînement (texte) |
| `y_test.csv` | Labels de test (texte) |
| `preprocessor_v1.pkl` | Objet sklearn sérialisé (pour prédiction en production) |
| `preprocessing_metadata_v1.json` | Métadonnées du preprocessing |


In [23]:
# ============================================================
# 10.1 — Sauvegarde des données et artefacts sklearn
# ============================================================

print("Sauvegarde des artefacts dans :", ARTIFACTS_DIR)
print()

# Dataset de travail nettoyé
df_model.to_csv(ARTIFACTS_DIR / "df_axe2_clean.csv", index=False)
print("df_axe2_clean.csv")

# Matrices preprocessées (pour le notebook 03)
X_train_df.to_csv(ARTIFACTS_DIR / "X_train_preprocessed.csv", index=True)
X_test_df.to_csv(ARTIFACTS_DIR / "X_test_preprocessed.csv", index=True)
print("X_train_preprocessed.csv")
print("X_test_preprocessed.csv")

# Labels (texte original, plus lisible)
y_train.to_csv(ARTIFACTS_DIR / "y_train.csv", index=True)
y_test.to_csv(ARTIFACTS_DIR / "y_test.csv", index=True)
print("y_train.csv")
print("y_test.csv")

# Preprocessor sérialisé (réutilisable pour Streamlit / production)
joblib.dump(preprocessor, ARTIFACTS_DIR / "preprocessor_v1.pkl")
print("preprocessor_v1.pkl")


Sauvegarde des artefacts dans : /content/drive/MyDrive/ML_Project/artifacts/axe2

df_axe2_clean.csv
X_train_preprocessed.csv
X_test_preprocessed.csv
y_train.csv
y_test.csv
preprocessor_v1.pkl


In [24]:
# ============================================================
# 10.2 — Sauvegarde des métadonnées du preprocessing
# ============================================================

metadata = {
    "version"           : "V1",
    "description"       : "Baseline complète — avec RCONSC et RDEF (leakage structurel intentionnel)",
    "dataset"           : "IST_corrected.csv",
    "n_patients_total"  : int(len(df)),
    "n_patients_model"  : int(len(df_model)),
    "n_train"           : int(X_train.shape[0]),
    "n_test"            : int(X_test.shape[0]),
    "n_features_before" : int(X.shape[1]),
    "n_features_after"  : int(X_train_df.shape[1]),
    "selected_features" : selected_features,
    "feature_names_out" : list(feature_names),
    "target"            : TARGET,
    "target_map"        : TARGET_MAP,
    "random_state"      : RANDOM_STATE,
    "test_size"         : 0.2,
    "stratified"        : True,
    "class_distribution": y.value_counts().to_dict(),
    "preprocessing" : {
        "numeric_strategy"     : "median imputation",
        "categorical_strategy" : "most_frequent imputation + OneHotEncoding",
        "rconsc_encoding"      : "ordinal (F=0, D=1, U=2)",
        "scaling"              : "none (V1 baseline)"
    },
    "notes" : [
        "V1 conserve intentionnellement RCONSC et RDEF pour démontrer le leakage structurel",
        "V2 exclura ces variables pour une évaluation réaliste",
        "RSBP=0 non traité en V1 (amélioration prévue en V2)",
        "C dans RDEF traité comme NaN (amélioration prévue en V2)"
    ]
}

with open(ARTIFACTS_DIR / "preprocessing_metadata_v1.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("preprocessing_metadata_v1.json")
print()
print("Résumé des métadonnées :")
print(f"  Version                  : {metadata['version']}")
print(f"  Patients dans le modèle  : {metadata['n_patients_model']:,}")
print(f"  Train / Test             : {metadata['n_train']:,} / {metadata['n_test']:,}")
print(f"  Features avant prepro    : {metadata['n_features_before']}")
print(f"  Features après prepro    : {metadata['n_features_after']}")


preprocessing_metadata_v1.json

Résumé des métadonnées :
  Version                  : V1
  Patients dans le modèle  : 19,418
  Train / Test             : 15,534 / 3,884
  Features avant prepro    : 20
  Features après prepro    : 47


In [25]:
# ============================================================
# 10.3 — Récapitulatif final de tous les artefacts sauvegardés
# ============================================================

print("=" * 60)
print("ARTEFACTS SAUVEGARDÉS DANS :", ARTIFACTS_DIR)
print("=" * 60)

for f in sorted(ARTIFACTS_DIR.glob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size_kb:8.1f} KB")


ARTEFACTS SAUVEGARDÉS DANS : /content/drive/MyDrive/ML_Project/artifacts/axe2
  X_test_preprocessed.csv                          748.5 KB
  X_train_preprocessed.csv                        2991.5 KB
  comparison_final_v1.csv                            0.2 KB
  confusion_matrix_v2.png                           77.5 KB
  df_axe2_clean.csv                               5788.6 KB
  feature_importance_v1.csv                          0.8 KB
  feature_importance_v2.png                        105.5 KB
  features_eda.json                                  0.6 KB
  model_metadata_v1.json                             0.9 KB
  modeling_summary_axe2.json                         0.3 KB
  models                                             4.0 KB
  pipeline_final_axe2.pkl                        12951.1 KB
  preprocessing_metadata.json                        1.4 KB
  preprocessing_metadata_v1.json                     2.3 KB
  preprocessor_axe2.pkl                              6.3 KB
  preprocessor_v1.pkl 

---
## Section 11 — Résumé et bilan du notebook

### 11.1 Ce qui a été accompli

| Étape | Action | Résultat |
|-------|--------|----------|
| Configuration | `config.py` centralisé + `axe2_utils.py` | Pipeline reproductible |
| Chargement | Dataset brut (19 435 patients, 112 colonnes) | Cohérence avec EDA garantie |
| Cible | Construction de `severity_class` | 3 classes : léger (50%) / modéré (42%) / sévère (8%) |
| Features | Version 1 : baseline complète (20 features) | RCONSC + RDEF inclus (leakage intentionnel) |
| Split | Train/Test 80/20 stratifié | Train : 15 534 / Test : 3 884 |
| Preprocessing | Imputation médiane + OneHot | 47 features après encodage, 0 NaN |
| Sauvegarde | 6 artefacts + métadonnées JSON | Notebook 03 autonome |

### 11.2 Points d'attention pour le notebook 03

1. **Leakage V1 vs V2** : la comparaison des performances V1/V2 montrera l'effet du leakage structurel
2. **Classe sévère (8%)** : les métriques par classe (F1-macro, recall sur severe) seront prioritaires
3. **F1-macro** est la métrique principale recommandée dans le cahier des charges pour cet axe

### 11.3 Améliorations prévues en Version 2

| Problème identifié | Solution prévue V2 |
|---|---|
| "C" dans RDEF traité comme NaN | Encodage 0.5 + indicateur `_uncertain` |
| Pas de StandardScaler | Ajout dans le sous-pipeline numérique |
| RSBP=0 non traité | Remplacement 0→NaN avant imputation |
| Déséquilibre non traité activement | SMOTE sur X_train uniquement |
| Pas de XGBoost | Ajout dans la comparaison modèles |

---

> **Ce notebook est prêt pour le notebook 03 — Modélisation V1.**  
> Tous les artefacts sont sauvegardés dans `artifacts/axe2/`.
